# 問題
問題75のパディングの処理を活用して、ミニバッチでモデルを学習せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [1]:
# 単語埋め込み語彙の作成
import numpy as np
from gensim.models import KeyedVectors
import torch

model = KeyedVectors.load_word2vec_format('./GoogleNews-vectors-negative300.bin', binary=True)
vocab = list(model.key_to_index.keys())
d_emb = model.vector_size
V = len(vocab) + 1

# 埋め込み行列の初期化
E = np.zeros((V, d_emb), dtype=np.float32)

# インデックス対応表
word2id = {'<PAD>': 0}
id2word = {0: '<PAD>'}

# 行列にベクトルを格納
for i, word in enumerate(vocab, start=1):
    E[i] = model[word]
    word2id[word] = i
    id2word[i] = word

In [3]:
import torch

def sst_build_answer_batch(path: str):
    """
    SST-2のTSVを読み込み、
      - 文章→単語分割
      - word2idでID列に変換（辞書にない語は除外）
      - 最長系列長に合わせて0埋めパディング
      - トークン列の長い順にソート
      - 最終的に input_ids と label をそれぞれTensorとしてまとめて返す
    """
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            row = raw.strip().split("\t")
            if len(row) < 2:
                continue
            label = row[1]
            if label not in ("0", "1"):
                continue
            text = row[0]
            tokens = text.strip().split()
            examples.append({"text": text, "label": int(label), "tokens": tokens})

    # ID化
    kept_examples = []
    for ex in examples:
        ids = [word2id[w] for w in ex["tokens"] if w in word2id]
        if ids:
            ex["input_ids"] = ids
            kept_examples.append(ex)

    # --- 長い順にソート ---
    kept_examples = sorted(kept_examples, key=lambda x: len(x["input_ids"]), reverse=True)

    # --- パディング ---
    max_len = len(kept_examples[0]["input_ids"])
    padded_ids = []
    labels = []

    for ex in kept_examples:
        ids = ex["input_ids"]
        padded = ids + [0] * (max_len - len(ids))
        padded_ids.append(padded)
        labels.append([float(ex["label"])])  # [[1.], [0.], ...]

    # --- Tensor化 ---
    input_tensor = torch.tensor(padded_ids, dtype=torch.long)
    label_tensor = torch.tensor(labels, dtype=torch.float)

    # --- dict形式で返す ---
    batch = {
        "input_ids": input_tensor,
        "label": label_tensor
    }

    return batch
# --- pathの準備 ---
path_dev = "./SST-2/dev.tsv"
path_train = "./SST-2/train.tsv"
dev_71 = sst_build_answer_batch(path_dev)
train_71 = sst_build_answer_batch(path_train)

In [5]:
import torch
import torch.nn.functional as F
import numpy as np

def build_avg_features(batch, E, pad_id=0):
    """
    batch: {"input_ids": LongTensor (N, L), "label": FloatTensor (N, 1)}
    E    : (|V|, d)  埋め込み行列（torch.Tensor or np.ndarray）
    戻り値: X (N, d), y (N,)
    """
    # E を torch.Tensor(float32) に揃える
    if isinstance(E, np.ndarray):
        E = torch.tensor(E, dtype=torch.float32)
    elif not torch.is_tensor(E):
        raise TypeError("E must be a torch.Tensor or np.ndarray")
    if E.dtype != torch.float32:
        E = E.float()

    ids = batch["input_ids"]            # (N, L) long
    y   = batch["label"].squeeze(1)     # (N,) float

    # (N, L, d) に埋め込み
    emb = F.embedding(ids, E)           # = E[ids] と同等

    # PADを無視して平均
    mask = (ids != pad_id)              # (N, L) bool
    lens = mask.sum(dim=1, keepdim=True).clamp_min(1)  # (N, 1)

    emb_sum = (emb * mask.unsqueeze(-1).float()).sum(dim=1)  # (N, d)
    X = emb_sum / lens.float()                                # (N, d)

    return X, y

X_train, y_train = build_avg_features(train_71, E)
X_dev, y_dev = build_avg_features(dev_71,   E)

In [6]:
import torch
from torch import nn

model = nn.Linear(X_train.size(1), 1)
crit  = nn.BCEWithLogitsLoss()
opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# ラベルをfloatに（BCEWithLogitsLoss用）
y_train_f = y_train.float()
y_dev_f   = y_dev.float()

epochs = 10
batch_size = 36

for epoch in range(1, epochs+1):
    model.train()
    # シャッフル
    idx = torch.randperm(X_train.size(0))
    X_train_shuf = X_train[idx]
    y_train_shuf = y_train_f[idx]

    running_loss = 0.0
    for i in range(0, X_train_shuf.size(0), batch_size):
        xb = X_train_shuf[i:i+batch_size]
        yb = y_train_shuf[i:i+batch_size]

        opt.zero_grad()
        logit = model(xb).squeeze(1)               # (B,)
        loss  = crit(logit, yb)
        loss.backward()
        opt.step()

        running_loss += loss.item() * xb.size(0)

    # ---- エポック終わりにログ ----
    train_loss = running_loss / X_train_shuf.size(0)

    model.eval()
    with torch.no_grad():
        logit_dev = model(X_dev).squeeze(1)
        prob_dev  = torch.sigmoid(logit_dev)
        pred_dev  = (prob_dev >= 0.5).long()
        acc_dev   = (pred_dev == y_dev.long()).float().mean().item()

    print(f"[Epoch {epoch:02d}] train loss: {train_loss:.4f}  |  dev acc: {acc_dev:.3f}")

# 学習後：重み・バイアス
w = model.weight.detach().squeeze(0)  # (d,)
b = model.bias.detach().item()

[Epoch 01] train loss: 0.5013  |  dev acc: 0.772
[Epoch 02] train loss: 0.4110  |  dev acc: 0.780
[Epoch 03] train loss: 0.3919  |  dev acc: 0.791
[Epoch 04] train loss: 0.3839  |  dev acc: 0.787
[Epoch 05] train loss: 0.3796  |  dev acc: 0.795
[Epoch 06] train loss: 0.3769  |  dev acc: 0.791
[Epoch 07] train loss: 0.3752  |  dev acc: 0.796
[Epoch 08] train loss: 0.3739  |  dev acc: 0.798
[Epoch 09] train loss: 0.3731  |  dev acc: 0.799
[Epoch 10] train loss: 0.3723  |  dev acc: 0.795
